# Environment / Dependencies Setup

| Package | Purpose |
|---------|---------|
| `pageindex` | Builds the document tree and runs the search |
| `pymupdf` | Extracts raw text from PDF pages |
| `langchain-openai` | `ChatOpenAI`, an OpenAI-compatible client we point at OpenRouter |
| `langchain-core` | The `HumanMessage` type |

In [ ]:
!pip install -q pageindex pymupdf langchain-core langchain-openai

## Import Libraries

In [ ]:
import json
import os
import re
import time

import pymupdf
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from pageindex import PageIndexClient, utils

## Add Your Keys

Each cell reads the key from an environment variable if it exists and otherwise asks you to paste it.

In [ ]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key (get one at https://openrouter.ai): ").strip()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
if not PAGEINDEX_API_KEY:
    PAGEINDEX_API_KEY = input("Enter your PageIndex API key (get one at https://pageindex.ai): ").strip()
os.environ["PAGEINDEX_API_KEY"] = PAGEINDEX_API_KEY

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
print("Keys loaded.")

**What you should see:** `Keys loaded.`

## Set Up the LLM

`call_llm(prompt)` sends one prompt and returns the reply text. It is used in every part. `temperature=0` keeps answers repeatable. The model is a *reasoning* model, so `extra_body={"reasoning": {"enabled": False}}` turns its hidden thinking off; without it, it can spend the whole `max_tokens` budget thinking and return an empty answer.

In [ ]:
llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0,
    max_tokens=512,
    extra_body={"reasoning": {"enabled": False}},
)

def call_llm(prompt):
    return llm.invoke([HumanMessage(content=prompt)]).content.strip()

# Step-wise Instructions — Development


### Setup — Index the PDF Once

The PDF is bundled in `data/`. We send it to PageIndex, wait until the tree is ready, and fetch it. This is the slow step (it can take a minute or two), so it happens **once**; Parts 1, 2 and 3 all reuse `doc_id` and `tree`.

In [ ]:
PDF_PATH = os.path.join("data", "CCS-Q1-2025-Earnings-Release.pdf")
if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"Could not find '{PDF_PATH}'. Run the notebook from the lab folder.")

doc_id = pi_client.submit_document(PDF_PATH)["doc_id"]
print(f"Submitted: {doc_id}")

elapsed = 0
while not pi_client.is_retrieval_ready(doc_id):
    if elapsed >= 600:
        raise TimeoutError("PageIndex timeout")
    time.sleep(5)
    elapsed += 5
print(f"Tree ready after about {elapsed}s.")

tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
utils.print_tree(tree)

**What you should see:** a tree with one root (`0000`, pages 1-11) and children such as `First Quarter 2025 Highlights`, `Balance Sheet and Liquidity`, `Non-GAAP Financial Measures` and `Forward-Looking Statements`, which in many runs has sub-nodes for the report's tables (`Net New Home Contracts`, `Home Deliveries`, `Adjusted Net Income ...`, `EBITDA and Adjusted EBITDA`, and so on). Each node has a summary. The exact tree can vary slightly between runs.


### Part 1 — Reasoning Retrieval: Let the LLM Pick the Section

**The idea:** give the LLM the tree (titles and summaries only) and ask which sections hold the answer. Then read the text of those pages and answer from it. No vectors anywhere.

```mermaid
flowchart LR
    Q["Question"] --> P["LLM reads the tree<br/>(titles + summaries)"]
    P --> N["node_list + reasoning"]
    N --> M["node_id -> page numbers"]
    M --> X["Read those pages<br/>with PyMuPDF"]
    X --> A["LLM answers from<br/>that text only"]
```

First, the search step. The prompt demands JSON, and `parse_json` has a fallback for when the LLM wraps the JSON in extra words:

In [ ]:
def parse_json(reply):
    try:
        return json.loads(reply)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", reply, re.DOTALL)
        return json.loads(match.group()) if match else {"thinking": "", "node_list": []}

def pick_sections(query):
    prompt = f"""You MUST respond with valid JSON only. No other text.

You are given a question and a document tree. Each node has: node_id, title, summary.
Pick the most specific nodes likely to contain the answer.

Question: {query}

Document tree:
{json.dumps(tree, indent=1)}

Respond with ONLY this JSON:
{{"thinking": "<your reasoning>", "node_list": ["node_id_1", "node_id_2"]}}"""
    return parse_json(call_llm(prompt))

Next, turn the chosen `node_id`s into page text. `node_map` looks up each node's page range; `page_texts` holds the raw text of every page. We read at most 3 pages per node so one huge node cannot flood the prompt.

In [ ]:
node_map = utils.create_node_mapping(tree, include_page_ranges=True)

doc = pymupdf.open(PDF_PATH)
page_texts = {i + 1: doc.load_page(i).get_text() for i in range(len(doc))}
doc.close()

def build_context(node_ids, max_pages=3):
    pages = []
    for nid in node_ids:
        if nid in node_map:
            info = node_map[nid]
            span = range(info["start_index"], info["end_index"] + 1)
            pages += [p for p in list(span)[:max_pages] if p not in pages]
    context = "\n\n".join(f"--- Page {p} ---\n{page_texts[p]}" for p in pages)
    return context, pages

Finally, the whole of Part 1 in one function, then run it:

In [ ]:
def reasoning_rag(query):
    picked = pick_sections(query)
    print("Reasoning:", picked.get("thinking", "")[:300])
    for nid in picked.get("node_list", []):
        title = node_map[nid]["node"]["title"] if nid in node_map else "not in tree"
        print(f"  picked {nid} | {title}")
    context, pages = build_context(picked.get("node_list", []))
    if not context:
        return "No relevant section found.", pages
    answer = call_llm(f"""Answer using only the context. If the answer is not there, say so. Be concise.

Context:
{context}

Question: {query}""")
    return answer, pages

QUERY_1 = "How much cash and total liquidity did the company have at the end of the first quarter?"
print("Question:", QUERY_1, "\n")
answer_1, pages_1 = reasoning_rag(QUERY_1)
print("\nPages read:", pages_1)
print("Answer:", answer_1)

**What you should see:** the LLM's reasoning, a pick such as `0003 | Balance Sheet and Liquidity` (page 2), and a short answer quoting the cash and liquidity figures from the document. You can read *why* it picked that node.


### Part 2 — Multi-Hop Retrieval: Collect From Several Sections

**The idea:** some questions need facts from more than one place. Instead of stopping at one section, we ask PageIndex's search for the top several matching sections and visit them one after another (each is a **hop**). For every hop we keep its title, `node_id`, pages and text, so we can show a trail afterward.

```mermaid
flowchart LR
    Q["Question"] --> S["PageIndex search"]
    S --> H1["Hop 1: section"]
    S --> H2["Hop 2: section"]
    S --> H3["Hop 3: section"]
    H1 --> C["Combine all text"]
    H2 --> C
    H3 --> C
    C --> A["LLM answer"]
    H1 --> E["Trace: which sections,<br/>which pages, why"]
    H2 --> E
    H3 --> E
```

`retrieve` submits the question, waits for the search to finish (retrying on temporary hiccups such as rate limits), and turns each returned node into a small record. PageIndex puts the page inside a string like `"<physical_index_6>"`, so a tiny regex pulls out the number. Parts 2 and 3 both use this function.

In [ ]:
def retrieve(query, top_k=5):
    retrieval_id = pi_client.submit_query(doc_id=doc_id, query=query).get("retrieval_id")
    if not retrieval_id:
        return []

    nodes = []
    for _ in range(60):
        try:
            result = pi_client.get_retrieval(retrieval_id)
            if result.get("status") == "completed":
                nodes = result.get("retrieved_nodes", [])
                break
            if result.get("status") == "failed":
                return []
        except Exception as e:
            print(f"Temporary problem while polling, retrying... ({e})")
        time.sleep(3)

    items = []
    for i, node in enumerate(nodes[:top_k]):
        texts, pages = [], []
        for group in node.get("relevant_contents", []):
            for piece in group:
                if piece.get("relevant_content"):
                    texts.append(piece["relevant_content"])
                match = re.search(r"(\d+)", str(piece.get("physical_index", "")))
                if match and int(match.group(1)) not in pages:
                    pages.append(int(match.group(1)))
        items.append({"number": i + 1, "section": node.get("title") or f"Section {i + 1}",
                      "node_id": node.get("id", "unknown"), "pages": pages, "text": "\n".join(texts)})
    return items

`explain` prints the trail for any list of retrieved items: where each came from, whether the answer used it, and a short LLM-written reason. The `is_used` argument is a function so each part can plug in its own check.

In [ ]:
def explain(query, items, is_used, kind="Hop"):
    print("\n--- EXPLAINABILITY ---")
    for it in items:
        pages = ", ".join(map(str, it["pages"])) or "unknown"
        status = "USED in answer" if is_used(it) else "retrieved but NOT clearly used"
        print(f'\n{kind} {it["number"]}: "{it["section"]}"')
        print(f'node_id: {it["node_id"]} | page(s): {pages} | {status}')
        why = call_llm(f"""In 3-4 short lines, explain why the section below is relevant to the question.
Mention the actual numbers or facts in it. Do not repeat the question.

Question: {query}

Section title: {it["section"]}
Section content: {it["text"]}""")
        print("Why:", why)

Now the multi-hop pipeline. The "was it used" check here is a rough heuristic: does the section's title appear in the answer? It is only a guide; Part 3 replaces it with an exact check.

In [ ]:
def multi_hop_rag(query, top_k=5):
    hops = retrieve(query, top_k)
    if not hops:
        return "No relevant context found.", []
    context = "\n\n".join(h["text"] for h in hops)
    answer = call_llm(f"""You are a financial analyst. Answer using only the context below, in plain language,
the way you would explain it in a meeting. Be concise.

Context:
{context}

Question: {query}""")
    return answer, hops

QUERY_2 = ("How much did GAAP net income grow or shrink from Q1 2024 to Q1 2025? "
           "Separately, what was adjusted net income (not adjusted EBITDA) for both periods, "
           "and does it tell a different story?")
print("Question:", QUERY_2, "\n")
answer_2, hops = multi_hop_rag(QUERY_2)

print("--- SECTIONS SEARCHED ---")
for h in hops:
    print(f"Hop {h['number']}: {h['section']}")
print("\n--- ANSWER ---")
print(answer_2)

explain(QUERY_2, hops, is_used=lambda h: h["section"] in answer_2, kind="Hop")

**What you should see:** up to five hops (for example the results text, the highlights, and a larger section holding the adjusted-net-income table), an answer giving GAAP net income for both quarters (about $64.3 million down to $39.4 million) and adjusted net income for both (about $71.4 million down to $42.2 million), and the conclusion that both tell the same story. Below it, a trail with one entry per hop. Some hops may say they lack the numbers, and because the title-in-answer check is crude, even the hop that supplied the numbers can be marked `retrieved but NOT clearly used`. That is the honest trail of a search that cast a wide net, and it is exactly why Part 3 adds exact citations.


### Part 3 — Table Retrieval: Whole Tables With `[Table n]` Citations

**The idea:** a table lookup needs the table intact — headers, rows and footnotes together. PageIndex keeps tables as whole nodes, so `retrieve` already hands them back whole. What is new here is **proof**: we label each table `[Table 1]`, `[Table 2]` in the context and instruct the LLM to tag every number with its table. Then "was this table used?" is an exact check — is the tag in the answer?

```mermaid
flowchart LR
    Q["Question"] --> R["retrieve(top_k=2)<br/>whole tables"]
    R --> L["Label: [Table 1 - title],<br/>[Table 2 - title]"]
    L --> A["LLM answers and tags<br/>every number [Table n]"]
    A --> C{"Is [Table n]<br/>in the answer?"}
    C -->|Yes| U["USED"]
    C -->|No| N["retrieved but NOT used"]
```

The citation rule lives in its own prompt so you can read it on its own:

In [ ]:
table_prompt = """You are a financial data analyst. Answer ONLY from the text and tables below.
Pay strict attention to table rows, columns and footnotes. Do not round numbers unless asked.

CRITICAL RULES:
- Tag every number you use with its table, like this: [Table 1]
- If you use numbers from more than one table, tag each one separately.
- If the data is not in the text, say "Not found in document."

Context:
{context}

Question: {query}"""

def table_rag(query, top_k=2):
    tables = retrieve(query, top_k)
    if not tables:
        return "No relevant context found.", []
    context = "\n\n".join(f"[Table {t['number']} - {t['section']}]\n{t['text']}" for t in tables)
    return call_llm(table_prompt.format(context=context, query=query)), tables

QUERY_3 = "What was the total revenue for the three months ended March 31, 2025?"
print("Question:", QUERY_3, "\n")
answer_3, tables = table_rag(QUERY_3)

print("--- TABLES SEARCHED ---")
for t in tables:
    print(f"Table {t['number']}: {t['section']}")
print("\n--- ANSWER ---")
print(answer_3)

explain(QUERY_3, tables, is_used=lambda t: f"[Table {t['number']}]" in answer_3, kind="Table")

**What you should see:** two retrieved nodes (the search may return a whole results section or a table node, depending on the question), an answer of about `$903.2 million [Table 1]`, and a trail where the cited table is marked `USED in answer` and any uncited one is marked `retrieved but NOT clearly used`.


### Variations Worth Trying

Change one thing at a time and re-run the relevant part.

- **Part 1:** ask for a number that lives in a table (for example "How many homes were delivered?") and see whether the LLM picks a table node. Then swap the model in `llm` for another free OpenRouter model and compare which nodes it picks.
- **Part 2:** change `top_k` from 5 to 2, then to 8. Fewer hops can miss half the answer; more hops add noise.
- **Part 3:** ask "What was adjusted EBITDA for Q1 2025 and Q1 2024?" and check that the EBITDA table is retrieved whole and cited.


# What We Learnt

You built three kinds of retrieval on one PDF without a vector database, an embedding model or chunking.

**Key takeaways:**
- **PageIndex builds a tree** of sections and whole tables with summaries; it is indexed once and reused.
- **Reasoning retrieval** — an LLM reads titles and summaries, picks `node_id`s, and gives a readable reason.
- **Multi-hop** — questions that need several sections are answered by visiting several nodes and keeping a trail (title, `node_id`, pages).
- **Whole tables** — nothing is cut by word count, so numbers stay with their headers.
- **`[Table n]` citations** make "which table was used" an exact check instead of a guess.
- **Explanations are grounded, not official** — the "why" is written by the LLM from the retrieved text.